# Fase 3 — Pipeline ETL
**Asignatura:** Gestión de Datos — UAX  
**Alumno:** Álvaro González Fernández

Carga del Data Warehouse `saleshealth_dwh.db` (SQLite) a partir de la BD transaccional `saleshealth` (PostgreSQL).

**Orden de carga** (respeta las dependencias de FK):

1. `dim_date`     — generada a partir del rango de `sale.sale_date`
2. `dim_customer` — extracción directa de `customer`
3. `dim_store`    — `store` ⊕ `city_zone` (LEFT JOIN por `postal_code`)
4. `dim_product`  — `product` ⊕ `central_product` ⊕ `brand` ⊕ `category` (LEFT JOIN por nombre)
5. `dim_offer`    — extracción directa de `offer`
6. `fact_sales`   — `sale_item` ⊕ `sale` ⊕ `central_product` (cálculo de `margin`, `date_id`)

## 0. Conexiones y utilidades

In [1]:
import os
import time
import sqlite3
import psycopg2
import pandas as pd
from sqlalchemy import create_engine

# ── Origen: PostgreSQL ───────────────────────────────────────────────────────
PG_PARAMS = {
    'host': 'localhost', 'port': 5432, 'dbname': 'saleshealth',
    'user': 'postgres',  'password': 'alvaro',
}
pg_engine = create_engine(
    f"postgresql+psycopg2://{PG_PARAMS['user']}:{PG_PARAMS['password']}"
    f"@{PG_PARAMS['host']}:{PG_PARAMS['port']}/{PG_PARAMS['dbname']}"
)

# ── Destino: SQLite (DWH) ────────────────────────────────────────────────────
DWH_PATH = '../02_dwh/saleshealth_dwh.db'
DDL_PATH = '../02_dwh/crear_dwh.sql'

# Recreamos el DWH desde cero a partir del DDL de la Fase 2
if os.path.exists(DWH_PATH):
    os.remove(DWH_PATH)

with open(DDL_PATH, 'r', encoding='utf-8') as f:
    ddl = f.read()

sqlite_conn = sqlite3.connect(DWH_PATH)
sqlite_conn.executescript(ddl)
sqlite_conn.execute('PRAGMA foreign_keys = ON;')
sqlite_conn.commit()

print(f'✓ DWH inicializado en {DWH_PATH}')
print(f'✓ Engine PostgreSQL conectado a {PG_PARAMS["dbname"]}@{PG_PARAMS["host"]}')

# ── Métricas del pipeline ────────────────────────────────────────────────────
metrics = []  # se rellena en cada step

def run_step(name, dwh_table, source_query, transform_fn=None):
    """Ejecuta una etapa del ETL en una transacción y registra métricas."""
    t0 = time.time()
    print(f'▶ [{name}] extracción…', end=' ', flush=True)
    try:
        df_src = pd.read_sql(source_query, pg_engine)
        n_extracted = len(df_src)
        print(f'{n_extracted:>6} filas │ transform…', end=' ', flush=True)

        df_dst = transform_fn(df_src) if transform_fn else df_src
        n_loaded = len(df_dst)

        # Carga en transacción explícita (rollback si falla)
        try:
            df_dst.to_sql(dwh_table, sqlite_conn,
                          if_exists='append', index=False)
            sqlite_conn.commit()
        except Exception as e:
            sqlite_conn.rollback()
            raise

        elapsed = round(time.time() - t0, 3)
        print(f'load {n_loaded:>6} ✓ ({elapsed}s)')
        metrics.append({
            'paso': name, 'tabla': dwh_table,
            'extraídas': n_extracted, 'cargadas': n_loaded,
            'tiempo (s)': elapsed,
        })
    except Exception as e:
        elapsed = round(time.time() - t0, 3)
        print(f'\n  ✗ ERROR tras {elapsed}s: {e}')
        metrics.append({
            'paso': name, 'tabla': dwh_table,
            'extraídas': -1, 'cargadas': 0,
            'tiempo (s)': elapsed,
        })
        raise

✓ DWH inicializado en ../02_dwh/saleshealth_dwh.db
✓ Engine PostgreSQL conectado a saleshealth@localhost


## 1. `dim_date` — calendario generado

In [2]:
def transform_date(df_range):
    dmin = pd.to_datetime(df_range.loc[0, 'dmin'])
    dmax = pd.to_datetime(df_range.loc[0, 'dmax'])
    rng = pd.date_range(dmin.normalize(), dmax.normalize(), freq='D')
    return pd.DataFrame({
        'date_id':     rng.strftime('%Y%m%d').astype(int),
        'date':        rng.strftime('%Y-%m-%d'),
        'year':        rng.year,
        'quarter':     rng.quarter,
        'month':       rng.month,
        'week':        rng.isocalendar().week.values,
        'day_of_week': rng.dayofweek + 1,                  # 1=lun … 7=dom
        'is_weekend':  (rng.dayofweek >= 5).astype(int),
    })

run_step(
    name='dim_date',
    dwh_table='dim_date',
    source_query='SELECT MIN(sale_date)::date AS dmin, MAX(sale_date)::date AS dmax FROM sale;',
    transform_fn=transform_date,
)

▶ [dim_date] extracción…      1 filas │ transform… load   2191 ✓ (0.387s)


## 2. `dim_customer`

In [3]:
def transform_customer(df):
    # Convertir timestamp a TEXT (formato ISO) para SQLite
    df = df.copy()
    df['created_at'] = pd.to_datetime(df['created_at']).dt.strftime('%Y-%m-%d %H:%M:%S')
    return df

run_step(
    name='dim_customer',
    dwh_table='dim_customer',
    source_query="""
        SELECT customer_id, first_name, last_name, email, phone, created_at
        FROM   customer
        ORDER  BY customer_id;
    """,
    transform_fn=transform_customer,
)

▶ [dim_customer] extracción…   5750 filas │ transform… load   5750 ✓ (0.176s)


## 3. `dim_store` ⊕ `city_zone`

In [4]:
run_step(
    name='dim_store',
    dwh_table='dim_store',
    source_query="""
        SELECT s.store_id,
               s.name,
               s.address,
               s.city,
               s.postal_code,
               cz.district,
               cz.area_type,
               cz.zone_orientation
        FROM   store s
        LEFT JOIN city_zone cz ON s.postal_code = cz.postal_code
        ORDER  BY s.store_id;
    """,
)

▶ [dim_store] extracción…     20 filas │ transform… load     20 ✓ (0.03s)


## 4. `dim_product` ⊕ `central_product` ⊕ `brand` ⊕ `category`

In [5]:
run_step(
    name='dim_product',
    dwh_table='dim_product',
    source_query="""
        SELECT p.product_id,
               p.name,
               p.category,
               p.manufacturer,
               cp.unit_cost,
               COALESCE(cp.unit_price, p.price) AS unit_price,
               b.name  AS brand_name,
               c.name  AS category_name
        FROM        product p
        LEFT JOIN   central_product cp ON p.name      = cp.name
        LEFT JOIN   brand           b  ON cp.brand_id = b.brand_id
        LEFT JOIN   category        c  ON cp.category_id = c.category_id
        ORDER BY    p.product_id;
    """,
)

▶ [dim_product] extracción…     50 filas │ transform… load     50 ✓ (0.033s)


## 5. `dim_offer`

In [6]:
def transform_offer(df):
    df = df.copy()
    for c in ('start_date', 'end_date'):
        df[c] = pd.to_datetime(df[c]).dt.strftime('%Y-%m-%d')
    return df

run_step(
    name='dim_offer',
    dwh_table='dim_offer',
    source_query="""
        SELECT offer_id, name, discount_percent, start_date, end_date
        FROM   offer
        ORDER  BY offer_id;
    """,
    transform_fn=transform_offer,
)

▶ [dim_offer] extracción…      1 filas │ transform… load      1 ✓ (0.028s)


## 6. `fact_sales`

Hechos al nivel de línea de venta. El cálculo:

$$\text{margin} = \text{subtotal} - \text{unit\_cost} \times \text{quantity}$$

El `unit_cost` se obtiene de `central_product` (catálogo central), enlazado a `product` por nombre. Si un producto no tiene match en `central_product`, se asume `unit_cost = 0` (se reporta en validación).

In [7]:
def transform_fact(df):
    df = df.copy()
    df['unit_cost'] = df['unit_cost'].fillna(0).astype(float)
    df['margin']    = df['subtotal'].astype(float) - df['unit_cost'] * df['quantity'].astype(int)
    # offer_id puede ser nullable: pandas usará NaN → SQLite lo guardará como NULL
    df['offer_id'] = df['offer_id'].astype('Int64')
    return df

run_step(
    name='fact_sales',
    dwh_table='fact_sales',
    source_query="""
        SELECT  si.sale_item_id,
                si.sale_id,
                s.customer_id,
                si.product_id,
                s.store_id,
                CAST(TO_CHAR(s.sale_date, 'YYYYMMDD') AS INTEGER) AS date_id,
                si.offer_id,
                si.quantity,
                si.unit_price,
                si.subtotal,
                cp.unit_cost
        FROM        sale_item si
        JOIN        sale            s  ON si.sale_id    = s.sale_id
        LEFT JOIN   product         p  ON si.product_id = p.product_id
        LEFT JOIN   central_product cp ON p.name        = cp.name
        ORDER BY    si.sale_item_id;
    """,
    transform_fn=transform_fact,
)

▶ [fact_sales] extracción…  42555 filas │ transform… load  42555 ✓ (1.442s)


## 7. Resumen del pipeline

In [8]:
df_metrics = pd.DataFrame(metrics)
print('─' * 60)
print('  RESUMEN DEL PIPELINE ETL')
print('─' * 60)
display(df_metrics)
print(f'\nTiempo total del pipeline: {df_metrics["tiempo (s)"].sum():.3f} s')
print(f'Filas totales cargadas:   {df_metrics["cargadas"].sum():,}')

────────────────────────────────────────────────────────────
  RESUMEN DEL PIPELINE ETL
────────────────────────────────────────────────────────────


,paso,tabla,extraídas,cargadas,tiempo (s)
0,dim_date,dim_date,1,2191,0.387
1,dim_customer,dim_customer,5750,5750,0.176
2,dim_store,dim_store,20,20,0.030
3,dim_product,dim_product,50,50,0.033
4,dim_offer,dim_offer,1,1,0.028
5,fact_sales,fact_sales,42555,42555,1.442



Tiempo total del pipeline: 2.096 s
Filas totales cargadas:   50,567


## 8. Validación del DWH

### 8.1. Recuento de filas en cada tabla

In [9]:
cur = sqlite_conn.cursor()
cur.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;")
tables = [r[0] for r in cur.fetchall()]

rows = []
for t in tables:
    cur.execute(f'SELECT COUNT(*) FROM "{t}";')
    rows.append({'tabla': t, 'filas en DWH': cur.fetchone()[0]})

df_counts = pd.DataFrame(rows)
display(df_counts)

,tabla,filas en DWH
0,dim_customer,5750
1,dim_date,2191
2,dim_offer,1
3,dim_product,50
4,dim_store,20
5,fact_sales,42555


### 8.2. Integridad referencial — FKs huérfanas en `fact_sales`

Comprobamos que ninguna FK del fact apunta a una clave inexistente en su dimensión.

In [10]:
checks = [
    ('customer_id', 'dim_customer', 'customer_id'),
    ('product_id',  'dim_product',  'product_id'),
    ('store_id',    'dim_store',    'store_id'),
    ('date_id',     'dim_date',     'date_id'),
    ('offer_id',    'dim_offer',    'offer_id'),
]

validation = []
for fk_col, dim_table, dim_pk in checks:
    # offer_id es nullable → excluimos NULL del recuento de huérfanos
    null_filter = f'AND f.{fk_col} IS NOT NULL' if fk_col == 'offer_id' else ''
    cur.execute(f"""
        SELECT COUNT(*) FROM fact_sales f
        LEFT JOIN {dim_table} d ON f.{fk_col} = d.{dim_pk}
        WHERE d.{dim_pk} IS NULL {null_filter};
    """)
    orphans = cur.fetchone()[0]
    validation.append({
        'FK del fact':    fk_col,
        'Dimensión':      dim_table,
        'huérfanas':      orphans,
        'estado':         '✓ OK' if orphans == 0 else '✗ FAIL',
    })

df_validation = pd.DataFrame(validation)
display(df_validation)

if (df_validation['huérfanas'] == 0).all():
    print('\n✓ Integridad referencial OK — no hay FKs huérfanas en fact_sales')
else:
    print('\n✗ Atención: existen FKs huérfanas. Revisar el ETL.')

,FK del fact,Dimensión,huérfanas,estado
0,customer_id,dim_customer,0,✓ OK
1,product_id,dim_product,0,✓ OK
2,store_id,dim_store,0,✓ OK
3,date_id,dim_date,0,✓ OK
4,offer_id,dim_offer,0,✓ OK



✓ Integridad referencial OK — no hay FKs huérfanas en fact_sales


### 8.3. Verificación de calidad de datos

In [11]:
quality = []

# Hechos sin coste (unit_cost = 0 → product sin match en central_product)
cur.execute('SELECT COUNT(*) FROM fact_sales WHERE unit_cost = 0;')
quality.append({'check': 'fact_sales con unit_cost = 0 (sin match en central_product)',
                'valor': cur.fetchone()[0]})

# Líneas con margen negativo
cur.execute('SELECT COUNT(*) FROM fact_sales WHERE margin < 0;')
quality.append({'check': 'fact_sales con margen negativo',
                'valor': cur.fetchone()[0]})

# Productos sin marca tras el join
cur.execute('SELECT COUNT(*) FROM dim_product WHERE brand_name IS NULL;')
quality.append({'check': 'dim_product sin brand_name (sin match en central_product)',
                'valor': cur.fetchone()[0]})

# Tiendas sin información de zona
cur.execute('SELECT COUNT(*) FROM dim_store WHERE district IS NULL;')
quality.append({'check': 'dim_store sin district (sin match en city_zone)',
                'valor': cur.fetchone()[0]})

# Total económico vendido
cur.execute('SELECT ROUND(SUM(subtotal), 2), ROUND(SUM(margin), 2) FROM fact_sales;')
rev, mar = cur.fetchone()
quality.append({'check': 'Ingresos totales (€)', 'valor': rev})
quality.append({'check': 'Margen total (€)',    'valor': mar})

df_quality = pd.DataFrame(quality)
display(df_quality)

,check,valor
0,fact_sales con unit_cost = 0 (sin match en cen...,711.00
1,fact_sales con margen negativo,0.00
2,dim_product sin brand_name (sin match en centr...,1.00
3,dim_store sin district (sin match en city_zone),0.00
4,Ingresos totales (€),9678678.67
5,Margen total (€),3888777.02


## 9. Cierre de conexiones

In [12]:
cur.close()
sqlite_conn.close()
pg_engine.dispose()
print('✓ Conexiones cerradas. DWH listo para la Fase 4 (métricas).')

✓ Conexiones cerradas. DWH listo para la Fase 4 (métricas).
